In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from pathlib import Path
import numpy as np

In [3]:
# Load the dataset
df = pd.read_csv("Housing.csv")

In [4]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
# Remove rows where target price is missing or invalid
df = df.dropna(subset=["price"])
df = df[df["price"] > 0]

In [8]:
df.shape

(545, 13)

In [9]:
print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nStatistical summary:")
print(df.describe(include="all").T)


Missing values in each column:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

Data types:
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object

Statistical summary:
                  count unique             top freq            mean  \
price             545.0    NaN             NaN  NaN  4766729.247706   
area              545.0    NaN             NaN  NaN     5150.541284   
bedrooms          545.0    NaN             NaN  NaN        2.965138  

In [10]:

# ==========================================
# 4. SEPARATE FEATURES AND TARGET
# ==========================================

X = df.drop(columns=["price"])
y = df["price"]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

# Identify numerical and categorical features
numerical_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)


Features shape: (545, 12)
Target shape: (545,)

Numerical columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
Categorical columns: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']


C:\Users\hp\AppData\Local\Temp\ipykernel_11884\4152040734.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [11]:

# ==========================================
# 5. TRAIN-TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 436
Testing samples: 109


In [12]:

# ==========================================
# 6. DATA PREPROCESSING
# ==========================================

# Numerical: replace missing values with column median
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical: impute missing values and encode categories
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply appropriate transformations to each column type
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])


In [13]:

# ==========================================
# 7. BUILD AND TRAIN MODEL
# ==========================================

n_estimators = 100
random_state = 42

model_params = {
    "model_type": "RandomForestRegressor",
    "n_estimators": n_estimators,
    "random_state": random_state,
    "test_size": 0.2
}

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1
    ))
])

print("\nTraining model...")

model.fit(X_train, y_train)

print("Training completed successfully.")


Training model...
Training completed successfully.


In [14]:

# ==========================================
# 8. MODEL EVALUATION
# ==========================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n========== MODEL EVALUATION ==========")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R2   : {r2:.4f}")


========== MODEL EVALUATION ==========
MAE  : 1,025,824.38
RMSE : 1,404,074.82
R2   : 0.6100


In [15]:

# ==========================================
# 10. SAVE TRAINED PIPELINE
# ==========================================

model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "house_price_model.joblib"

joblib.dump(model, model_path)

print(f"\nModel saved successfully: {model_path}")
print("Training pipeline finished.")



Model saved successfully: models\house_price_model.joblib
Training pipeline finished.


In [16]:
from pathlib import Path

print("Jupyter is running from:", Path.cwd())
print("Model full path:", Path("models/house_price_model.joblib").resolve())
print("Model exists:", Path("models/house_price_model.joblib").exists())

Jupyter is running from: C:\Users\hp
Model full path: C:\Users\hp\models\house_price_model.joblib
Model exists: True
